# MedSigLIP ESI Text-Embedding Test

MedSigLIP does not generate an ESI level. It maps medical images and short text
to embeddings. This notebook freezes MedSigLIP, embeds each MIMIC-IV-ED triage
record, and trains a logistic-regression classifier to map those embeddings to
ESI levels 1-5.

The experiment uses 4,000 balanced training encounters and 500 balanced
test encounters. This tests whether the pretrained text representation is useful for this
dataset. It does not test MedSigLIP's primary medical-image capability and is
not a clinical validation.

## 1. Configuration

In [ ]:
from pathlib import Path
import gc
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

MODEL_ID = "google/medsiglip-448"
LOCAL_FILES_ONLY = True
BATCH_SIZE = 16
SEED = 42

if LOCAL_FILES_ONLY:
    os.environ["HF_HUB_OFFLINE"] = "1"


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "prepare_training_data.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from transformers import AutoModel, AutoProcessor

from src.baseline_utils import (
    ESI_LEVELS,
    compute_metrics,
    load_split,
    serialize_record,
    test_fingerprint,
)
from src.clinical_triage_utils import embed_texts_medsiglip

DATA_DIR = PROJECT_ROOT / "data" / "finetune_mimic_balanced_large"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "medsiglip"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32
print(f"Device: {device}")

## 2. Load the larger balanced split

The classifier trains on 4,000 encounters and is evaluated on a
patient-separated test set of 500 encounters. Each ESI class contributes the
same number of records.

In [ ]:
train = load_split(DATA_DIR / "train.jsonl")
test = load_split(DATA_DIR / "test.jsonl")

assert set(train["subject_id"]).isdisjoint(test["subject_id"])
fingerprint = test_fingerprint(test)

display(pd.DataFrame({
    "train": train["label"].value_counts().sort_index(),
    "test": test["label"].value_counts().sort_index(),
}).rename_axis("ESI level"))
print(f"Train records: {len(train)}")
print(f"Test records: {len(test)}")
print(f"Test fingerprint: {fingerprint}")

## 3. Create frozen MedSigLIP embeddings

Each row becomes a short sentence containing chief complaint, vitals, and
pain. MedSigLIP accepts at most 64 text tokens, so the notebook reports how
often records exceed that limit.

In [ ]:
train_texts = [serialize_record(row) for _, row in train.iterrows()]
test_texts = [serialize_record(row) for _, row in test.iterrows()]

load_started = time.perf_counter()
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    local_files_only=LOCAL_FILES_ONLY,
)
model = AutoModel.from_pretrained(
    MODEL_ID,
    local_files_only=LOCAL_FILES_ONLY,
    dtype=dtype,
).to(device)
model.eval()
load_seconds = time.perf_counter() - load_started

all_texts = train_texts + test_texts
token_ids = processor.tokenizer(
    all_texts,
    add_special_tokens=True,
    truncation=False,
)["input_ids"]
token_lengths = np.array([len(ids) for ids in token_ids])
truncation_rate = float((token_lengths > 64).mean())

embedding_started = time.perf_counter()
train_embeddings = embed_texts_medsiglip(
    train_texts,
    processor,
    model,
    batch_size=BATCH_SIZE,
    desc="Training embeddings",
)
test_embeddings = embed_texts_medsiglip(
    test_texts,
    processor,
    model,
    batch_size=BATCH_SIZE,
    desc="Test embeddings",
)
embedding_seconds = time.perf_counter() - embedding_started

print(f"Embedding dimensions: {train_embeddings.shape[1]}")
print(f"Maximum tokens before truncation: {token_lengths.max()}")
print(f"Records over 64 tokens: {truncation_rate:.1%}")
print(f"Model load: {load_seconds:.1f} seconds")
print(f"Embedding: {embedding_seconds:.1f} seconds")

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Train the ESI classifier

MedSigLIP remains frozen. Only the small logistic-regression classifier learns
from the 500 labeled training records.

In [ ]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(max_iter=2000, random_state=SEED)

fit_started = time.perf_counter()
classifier.fit(train_embeddings, train["label"])
fit_seconds = time.perf_counter() - fit_started

predict_started = time.perf_counter()
probabilities = classifier.predict_proba(test_embeddings)
predictions = classifier.classes_[probabilities.argmax(axis=1)].astype(int)
prediction_scores = probabilities.max(axis=1)
predict_seconds = time.perf_counter() - predict_started

metrics = compute_metrics(test["label"], predictions)
print(f"Classifier fit: {fit_seconds:.3f} seconds")
print(f"Classifier prediction: {predict_seconds:.3f} seconds")

## 5. Results

In [ ]:
scalar_metrics = {
    key: value
    for key, value in metrics.items()
    if key not in {"recall_by_esi", "confusion_matrix_labels_1_to_5"}
}
display(pd.DataFrame([scalar_metrics]).round(3))

display(pd.DataFrame(
    metrics["confusion_matrix_labels_1_to_5"],
    index=[f"actual {level}" for level in ESI_LEVELS],
    columns=[f"predicted {level}" for level in ESI_LEVELS],
))

display(pd.DataFrame.from_dict(
    metrics["recall_by_esi"],
    orient="index",
    columns=["recall"],
).rename_axis("ESI level").round(3))

## 6. Exploratory review thresholds

The maximum logistic-regression probability can be used as a review score,
but it is not calibrated clinical confidence. This table is descriptive only;
a deployed threshold must be selected on validation data and tested on a
larger untouched cohort.

In [ ]:
actual = test["label"].to_numpy(dtype=int)
rows = []
for threshold in (0.0, 0.4, 0.5, 0.6):
    accepted = prediction_scores >= threshold
    accepted_count = int(accepted.sum())
    rows.append({
        "minimum_score": threshold,
        "coverage": float(accepted.mean()),
        "accepted_cases": accepted_count,
        "accuracy_when_accepted": (
            float((predictions[accepted] == actual[accepted]).mean())
            if accepted_count else None
        ),
        "severe_under_triage_when_accepted": (
            float((
                np.isin(actual[accepted], [1, 2])
                & np.isin(predictions[accepted], [4, 5])
            ).mean())
            if accepted_count else None
        ),
    })

review_table = pd.DataFrame(rows)
display(review_table.round(3))

## 7. Save aggregate results

In [ ]:
result = {
    "model_id": MODEL_ID,
    "method": "frozen_text_embeddings_with_logistic_regression",
    "test_fingerprint": fingerprint,
    "train_examples": len(train),
    "test_examples": len(test),
    "embedding_dimensions": int(train_embeddings.shape[1]),
    "maximum_text_tokens": 64,
    "truncation_rate": truncation_rate,
    "load_seconds": load_seconds,
    "embedding_seconds": embedding_seconds,
    "classifier_fit_seconds": fit_seconds,
    "classifier_predict_seconds": predict_seconds,
    "metrics": metrics,
    "review_thresholds": rows,
    "predictions": [
        {
            "actual": int(label),
            "prediction": int(prediction),
            "score": float(score),
        }
        for label, prediction, score in zip(actual, predictions, prediction_scores)
    ],
}

output_path = OUTPUT_DIR / "medsiglip_text_probe_large_balanced.json"
output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(f"Saved {output_path.relative_to(PROJECT_ROOT)}")